# Quickstart, correctness, and performance

## Goal

Tour the core API on a small readable graph, verify PageRank against
NetworkX, and then compare NetworkX with metal-graph's CPU and GPU paths on
the same larger graph.

The performance graph is seeded, synthetic, and self-contained. Both graph
objects are built before timing; reported values cover warm PageRank calls
through their returned Python results. The notebook runs in well under a
minute on an Apple Silicon Mac.

In [1]:
import numpy as np
import metal_graph as mg

print("metal-graph", mg.__version__, "| Metal GPU available:", mg.has_gpu())

metal-graph 0.1.0 | Metal GPU available: True


## 1. Build a graph from string IDs

`from_edges` accepts int32/int64/str IDs and maps them to dense **user
indices** `0..V-1` (`np.unique` order). Every algorithm speaks user indices;
`G.external_ids` and `G.index_of` translate both ways.

In [2]:
src = np.array(["alice", "alice", "bob",  "carol", "dave", "erin",
                "frank", "grace", "carol"])
dst = np.array(["bob",   "carol", "carol", "dave",  "alice", "frank",
                "erin",  "erin",  "alice"])
G = mg.Graph.from_edges(src, dst, directed=True)

print("V =", G.num_vertices, "| E =", G.num_edges)
print("external_ids:", G.external_ids.tolist())
print("index_of(['carol', 'erin']):", G.index_of(["carol", "erin"]).tolist())

V = 7 | E = 9
external_ids: ['alice', 'bob', 'carol', 'dave', 'erin', 'frank', 'grace']
index_of(['carol', 'erin']): [2, 4]


## 2. Check PageRank against NetworkX

Same iteration, same dangling handling, same weighted normalization. The
project's broader 1,000+ case suite covers correctness, properties, and
integration behavior; here is a direct NetworkX spot check.

In [3]:
import networkx as nx

pr = np.asarray(mg.pagerank(G, alpha=0.85, tol=1e-10, max_iter=200))

nxg = nx.DiGraph()
nxg.add_nodes_from(G.external_ids)
nxg.add_edges_from(zip(src, dst))
nx_pr = nx.pagerank(nxg, alpha=0.85, tol=1e-10, max_iter=200)
nx_vec = np.array([nx_pr[e] for e in G.external_ids])

print("max absolute PageRank score difference =",
      float(np.abs(pr - nx_vec).max()))
top = np.argsort(-pr)[:3]
print("top-3:", [(str(G.external_ids[i]), round(float(pr[i]), 4)) for i in top])

max absolute PageRank score difference = 7.13313735856147e-09
top-3: [('erin', 0.2085), ('frank', 0.1986), ('alice', 0.1855)]


## 3. Run BFS, k-hop, and connected components

In [4]:
alice = int(G.index_of("alice"))
dist, parent = mg.bfs(G, sources=[alice], direction="out")
print("BFS depths from alice:",
      {str(G.external_ids[i]): int(d) for i, d in enumerate(dist) if d >= 0})

vs, es = mg.k_hop(G, seeds=[alice], k=2, direction="both")
print("2-hop neighborhood:", [str(G.external_ids[v]) for v in vs])
sub = mg.k_hop(G, seeds=[alice], k=2, direction="both", as_graph=True)
print("materialized subgraph: V =", sub.num_vertices, "E =", sub.num_edges)

comp = np.asarray(mg.experimental.wcc(G))
print("weakly connected components:", int(comp.max()) + 1)

BFS depths from alice: {'alice': 0, 'bob': 1, 'carol': 1, 'dave': 2}
2-hop neighborhood: ['alice', 'bob', 'carol', 'dave']
materialized subgraph: V = 4 E = 6
weakly connected components: 2


## 4. See why graph size matters

This seven-vertex graph is intentionally too small for GPU acceleration.
Forcing both paths demonstrates launch overhead, not throughput: automatic
mode keeps work like this on the CPU. `last_run_info()` reports the path
that actually executed.

In [5]:
for mode in (["cpu", "gpu"] if mg.has_gpu() else ["cpu"]):
    mg.set_execution(mode)
    mg.pagerank(G, alpha=0.85, tol=1e-8, max_iter=100)
    print(f"mode={mode!r:6} ->", mg.last_run_info())
mg.set_execution("auto")

mode='cpu'  -> {'op': 'pagerank', 'path': 'cpu', 'iterations': 95, 'ms': 0.078208}
mode='gpu'  -> {'op': 'pagerank', 'path': 'gpu', 'iterations': 95, 'ms': 16.392916}


## 5. Compare NetworkX, metal-graph CPU, and metal-graph GPU

A useful speed comparison needs a graph large enough to amortize GPU launch
cost. The cell below generates a deterministic RMAT-shaped directed graph,
removes duplicate edge pairs so NetworkX and metal-graph receive identical
simple-graph semantics, and builds both graph objects before timing.

The comparison uses `alpha=0.85`, `tol=1e-10`, and `max_iter=200`. Each
implementation gets one warm-up call; displayed values are medians of 5 CPU,
10 GPU, and 3 NetworkX calls. These are end-to-end PageRank API timings on
one machine, including any per-call preparation each API performs. NetworkX
does not expose its iteration count here, so this is a same-tolerance API
comparison rather than a fixed-iteration kernel benchmark. Treat it as an
illustration, not a universal speed guarantee; the repository's
[benchmark artifacts](../bench/README.md) contain the publication-grade
measurements.

In [6]:
import statistics
import time

def generate_simple_rmat(scale=16, edgefactor=20, seed=11):
    """Seeded RMAT edges, deduplicated to one edge per (src, dst)."""
    rng = np.random.default_rng(seed)
    n_edges = edgefactor << scale
    a, b, c = 0.57, 0.19, 0.19
    ab = a + b
    c_norm = c / (1.0 - ab)
    a_norm = a / ab
    src = np.zeros(n_edges, dtype=np.int64)
    dst = np.zeros(n_edges, dtype=np.int64)
    for bit in range(scale):
        row_bit = rng.random(n_edges) > ab
        col_bit = rng.random(n_edges) > (
            c_norm * row_bit + a_norm * (~row_bit)
        )
        src |= row_bit.astype(np.int64) << bit
        dst |= col_bit.astype(np.int64) << bit

    n_vertices = 1 << scale
    edge_keys = src.astype(np.uint64) * n_vertices + dst
    _, first = np.unique(edge_keys, return_index=True)
    first.sort()
    return (src[first].astype(np.uint32),
            dst[first].astype(np.uint32), n_vertices, n_edges)

perf_src, perf_dst, perf_vertices, candidate_edges = generate_simple_rmat()
perf_graph = mg.Graph.from_edges(
    perf_src, perf_dst, directed=True, num_vertices=perf_vertices
)

nx_perf_graph = nx.DiGraph()
nx_perf_graph.add_nodes_from(range(perf_vertices))
nx_perf_graph.add_edges_from(zip(perf_src.tolist(), perf_dst.tolist()))

print(f"benchmark graph: V={perf_vertices:,} E={len(perf_src):,} "
      f"({candidate_edges:,} seeded candidates before deduplication)")

benchmark graph: V=65,536 E=1,177,876 (1,310,720 seeded candidates before deduplication)


In [7]:
def median_call_ms(fn, runs):
    fn()  # warm-up
    samples = []
    result = None
    for _ in range(runs):
        started = time.perf_counter()
        result = fn()
        samples.append((time.perf_counter() - started) * 1_000)
    return float(statistics.median(samples)), result

rank_kwargs = dict(alpha=0.85, tol=1e-10, max_iter=200)

try:
    mg.set_execution("cpu")
    cpu_ms, cpu_rank = median_call_ms(
        lambda: mg.pagerank(perf_graph, **rank_kwargs), runs=5
    )
    cpu_iterations = mg.last_run_info()["iterations"]

    gpu_ms = gpu_rank = gpu_iterations = None
    if mg.has_gpu():
        mg.set_execution("gpu")
        gpu_ms, gpu_rank = median_call_ms(
            lambda: mg.pagerank(perf_graph, **rank_kwargs), runs=10
        )
        gpu_iterations = mg.last_run_info()["iterations"]
finally:
    mg.set_execution("auto")

networkx_ms, networkx_rank_dict = median_call_ms(
    lambda: nx.pagerank(nx_perf_graph, **rank_kwargs), runs=3
)
networkx_rank = np.fromiter(
    (networkx_rank_dict[i] for i in range(perf_vertices)),
    dtype=np.float64,
    count=perf_vertices,
)

print(f"metal-graph CPU : {cpu_ms:8.2f} ms  "
      f"({cpu_iterations} iterations)")
if gpu_ms is not None:
    print(f"metal-graph GPU : {gpu_ms:8.2f} ms  "
          f"({gpu_iterations} iterations)")
print(f"NetworkX        : {networkx_ms:8.2f} ms")
print()
print(f"metal-graph CPU is {networkx_ms / cpu_ms:.1f}x faster than NetworkX")
if gpu_ms is not None:
    print(f"metal-graph GPU is {cpu_ms / gpu_ms:.1f}x faster than "
          "metal-graph CPU")
    print(f"metal-graph GPU is {networkx_ms / gpu_ms:.1f}x faster than "
          "NetworkX")
    print("max absolute score difference (CPU vs GPU):",
          float(np.max(np.abs(np.asarray(cpu_rank) - np.asarray(gpu_rank)))))

comparison_rank = gpu_rank if gpu_rank is not None else cpu_rank
print("max absolute score difference (metal-graph vs NetworkX):",
      float(np.max(np.abs(np.asarray(comparison_rank) - networkx_rank))))

metal-graph CPU :     9.61 ms  (15 iterations)
metal-graph GPU :     5.25 ms  (15 iterations)
NetworkX        :   574.07 ms

metal-graph CPU is 59.7x faster than NetworkX
metal-graph GPU is 1.8x faster than metal-graph CPU
metal-graph GPU is 109.4x faster than NetworkX
max absolute score difference (CPU vs GPU): 1.3969838619232178e-09
max absolute score difference (metal-graph vs NetworkX): 4.879928381523735e-09


## Takeaways

- The small graph is the right place to learn the API and check semantics,
  but the wrong place to judge GPU throughput.
- On the 1,177,876-edge graph, the embedded M4 Max run measured 9.61 ms for
  metal-graph CPU, 5.25 ms for metal-graph GPU, and 574.07 ms for NetworkX.
  The GPU was 1.8x faster than metal-graph CPU; CPU was 59.7x faster than
  NetworkX; and GPU was 109.4x faster than NetworkX.
- CPU and GPU results differed by at most `1.40e-9`; metal-graph and
  NetworkX differed by at most `4.88e-9`.

## Next steps

Continue with [`02_batched_ppr_retrieval.ipynb`](02_batched_ppr_retrieval.ipynb), which
shows the flagship batched-PPR API on an agent-retrieval workload, and
[`03_bfs_latency_planner.ipynb`](03_bfs_latency_planner.ipynb) shows the
microsecond BFS latency path on a 2M-edge graph.